# AI Procurement Assistant

**Nub Labs - Manufacturing AI Series | Playbook 1**

Score every supplier on lead time, quality, and on-time delivery from 6 months of ERP data.

**Data source:** ForgeFlow Reference Platform (ERPNext v16 - FlowForge Industries)  
**Period:** January to June 2026  
**Suppliers scored:** 5 active vendors (8 total in master data)  
**Records:** 27 GRNs, 27 quality inspections, 32 purchase orders

**What this notebook builds:**
1. Lead time analysis per supplier (avg, min, max, variance)
2. On-time delivery rate per supplier
3. Quality rejection rate per supplier + monthly pattern detection
4. Composite supplier scorecard (quality 40% + on-time 35% + lead time 25%)
5. Automated reorder recommendations weighted by supplier grade

## 1. Load Data

All data loaded directly from the ForgeFlow GitHub repository - no local setup, no API key, no database connection.

The JSON files are clean flat exports from ERPNext v16 (see `forgeflow-reference-platform/data/seeds/export_json.py`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BASE = 'https://raw.githubusercontent.com/Nub-Labs/forgeflow-reference-platform/main/data'

suppliers_df   = pd.read_json(f'{BASE}/master/suppliers.json')
items_df       = pd.read_json(f'{BASE}/master/items.json')
pos_df         = pd.read_json(f'{BASE}/procurement/purchase_orders.json')
prs_df         = pd.read_json(f'{BASE}/procurement/purchase_receipts.json')
qi_df          = pd.read_json(f'{BASE}/quality/quality_inspections.json')

print(f'Suppliers:           {len(suppliers_df)}')
print(f'Purchase Orders:     {len(pos_df)}')
print(f'Purchase Receipts:   {len(prs_df)}')
print(f'Quality Inspections: {len(qi_df)}')
prs_df.head(3)

## 2. Lead Time Analysis

Lead time = days from purchase order date to goods receipt note (GRN) date.

The `purchase_receipts.json` export already includes `lead_days` (computed as `DATEDIFF(receipt_date, order_date)` in the ERPNext SQL export). We summarise per supplier.

In [ ]:
# Parse dates
prs_df['receipt_date'] = pd.to_datetime(prs_df['receipt_date'])
prs_df['order_date']   = pd.to_datetime(prs_df['order_date'])
prs_df['month']        = prs_df['receipt_date'].dt.month

# Lead time summary per supplier
lead_summary = (
    prs_df
    .groupby('supplier')['lead_days']
    .agg(avg_lead='mean', min_lead='min', max_lead='max', grn_count='count')
    .round(1)
    .sort_values('avg_lead')
    .reset_index()
)
print('Lead time summary (submitted GRNs, days):')
lead_summary

In [ ]:
# Short supplier names for charts
short_names = {
    'Amrit Metals Pvt Ltd':        'Amrit',
    'Purohit Metals Pvt Ltd':      'Purohit',
    'Sudarshan Metal Trading Co':  'Sudarshan',
    'Paschim Steel Corporation':   'Paschim',
    'Vishwakarma Tools & Dies':    'Vishwakarma',
}
lead_summary['short'] = lead_summary['supplier'].map(short_names)

fig, ax = plt.subplots()
colors = ['#10b981' if v <= 8 else '#f59e0b' if v <= 14 else '#ef4444'
          for v in lead_summary['avg_lead']]
bars = ax.barh(lead_summary['short'], lead_summary['avg_lead'], color=colors, height=0.5)
ax.set_xlabel('Average lead time (days)')
ax.set_title('Supplier Lead Time - 6 Month Average (Jan-Jun 2026)')
for bar, row in zip(bars, lead_summary.itertuples()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row.avg_lead}d (n={row.grn_count})', va='center', fontsize=9)
ax.set_xlim(0, 26)
patches = [mpatches.Patch(color='#10b981', label='Fast (<= 8d)'),
           mpatches.Patch(color='#f59e0b', label='Moderate (9-14d)'),
           mpatches.Patch(color='#ef4444', label='Slow (> 14d)')]
ax.legend(handles=patches, fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

**Reading:** Amrit (consumables) has the shortest lead time at 5.8 days. Vishwakarma (custom tooling) is at 21 days - long but expected for bespoke parts. Paschim Steel at 14 days is 40% slower than Purohit for equivalent steel categories.

## 3. On-Time Delivery Rate

On-time = GRN receipt date on or before the scheduled delivery date on the purchase order.

We need to join purchase receipts to purchase orders to get the scheduled date. The `po_id` in `purchase_receipts.json` provides this link.

In [ ]:
pos_df['order_date']    = pd.to_datetime(pos_df['order_date'])
pos_df['schedule_date'] = pd.to_datetime(pos_df['schedule_date'])

# Each PO has one item line in ForgeFlow - join on po_id to get schedule_date
# Use first occurrence per po_id since each PO has one receipt
po_schedule = pos_df[['po_id', 'schedule_date']].drop_duplicates('po_id')

prs_merged = prs_df.merge(po_schedule, on='po_id', how='left')

# Filter out June GRNs with no schedule_date (pending orders)
prs_scored = prs_merged.dropna(subset=['schedule_date'])

prs_scored = prs_scored.copy()
prs_scored['on_time'] = prs_scored['receipt_date'] <= prs_scored['schedule_date']

ontime_summary = (
    prs_scored
    .groupby('supplier')
    .agg(on_time_count=('on_time', 'sum'), total=('on_time', 'count'))
    .assign(on_time_pct=lambda d: (d['on_time_count'] / d['total'] * 100).round(1))
    .sort_values('on_time_pct', ascending=False)
    .reset_index()
)
ontime_summary['short'] = ontime_summary['supplier'].map(short_names)
print('On-time delivery rate per supplier:')
ontime_summary

In [ ]:
fig, ax = plt.subplots()
colors = ['#10b981' if v == 100 else '#f59e0b' if v >= 60 else '#ef4444'
          for v in ontime_summary['on_time_pct']]
bars = ax.barh(ontime_summary['short'], ontime_summary['on_time_pct'], color=colors, height=0.5)
ax.set_xlabel('On-time delivery rate (%)')
ax.set_title('Supplier On-Time Delivery Rate (Jan-Jun 2026)')
ax.set_xlim(0, 120)
for bar, row in zip(bars, ontime_summary.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row.on_time_pct}% ({int(row.on_time_count)}/{int(row.total)})',
            va='center', fontsize=9)
ax.axvline(100, color='#6b7280', linewidth=0.8, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Reading:** Paschim Steel has 0% on-time delivery - every single GRN arrived after the scheduled date. Sudarshan is at 60% (2 of 5 late). Both Amrit and Purohit are at 100%.

## 4. Quality Rejection Analysis

Quality inspections (incoming) are linked 1:1 to each GRN. We compute the rejection rate per supplier, then look at the monthly pattern to identify whether failures are random or systematic.

In [ ]:
qi_df['inspection_date'] = pd.to_datetime(qi_df['inspection_date'])
qi_df['month']           = qi_df['inspection_date'].dt.month
qi_df['rejected']        = qi_df['status'] == 'Rejected'

quality_summary = (
    qi_df
    .groupby('supplier')
    .agg(rejected_count=('rejected', 'sum'), total=('rejected', 'count'))
    .assign(rejection_pct=lambda d: (d['rejected_count'] / d['total'] * 100).round(1))
    .sort_values('rejection_pct', ascending=False)
    .reset_index()
)
quality_summary['short'] = quality_summary['supplier'].map(short_names)
print('Quality rejection rate per supplier:')
quality_summary

In [ ]:
# Monthly rejection pattern heatmap
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun'}

monthly_rejections = (
    qi_df
    .groupby(['supplier', 'month'])['rejected']
    .sum()
    .unstack(fill_value=0)
    .rename(columns=month_names)
)
monthly_rejections.index = [short_names.get(s, s) for s in monthly_rejections.index]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(monthly_rejections.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(monthly_rejections.columns)))
ax.set_xticklabels(monthly_rejections.columns)
ax.set_yticks(range(len(monthly_rejections.index)))
ax.set_yticklabels(monthly_rejections.index)
for i in range(len(monthly_rejections.index)):
    for j in range(len(monthly_rejections.columns)):
        val = monthly_rejections.values[i, j]
        label = 'FAIL' if val > 0 else 'PASS'
        ax.text(j, i, label, ha='center', va='center', fontsize=9,
                color='white' if val > 0 else '#374151', fontweight='bold' if val > 0 else 'normal')
ax.set_title('Monthly Rejection Pattern (Jan-Jun 2026)')
ax.set_xlabel('Month')
plt.tight_layout()
plt.show()

print('\nRejected batches detail:')
rejected = qi_df[qi_df['rejected']].copy()
rejected['month_name'] = rejected['month'].map(month_names)
print(rejected[['supplier', 'item_code', 'month_name', 'dimensional_reading', 'hardness_reading', 'status']].to_string(index=False))

**Reading:** The monthly pattern reveals that failures are not random:
- **Sudarshan Metal Trading Co** - rejections in Feb and Apr only (alternating months). Cause: EN24 billet hardness out-of-spec (readings: 69 HRB vs acceptable range 75-95). Points to a batch procurement cycle issue at the vendor.
- **Paschim Steel Corporation** - rejections in Mar and May only. Cause: dimensional deviation on SS304/AL6061 bar stock (readings: 0.06mm vs tolerance <= 0.04mm). Consistent failure mode pointing to tooling or process drift at their cutting facility.

A flat 40% rejection rate for each vendor understates the problem. The monthly pattern shows both vendors have recurring failures, not one-off quality events.

## 5. Composite Supplier Scorecard

Three weighted dimensions:
- **Quality score** (40%) = (1 - rejection_rate) x 100
- **On-time delivery score** (35%) = on_time_pct
- **Lead time score** (25%) = max(0, 1 - avg_lead_days/30) x 100

Grade thresholds: A (>= 80), B (>= 60), C (>= 40), D (< 40)

In [ ]:
# Build scorecard DataFrame
scorecard = lead_summary[['supplier', 'short', 'avg_lead', 'grn_count']].copy()

# Merge on-time
scorecard = scorecard.merge(
    ontime_summary[['supplier', 'on_time_pct']],
    on='supplier', how='left'
).fillna({'on_time_pct': 100.0})  # suppliers with no scored GRNs assumed 100%

# Merge quality
scorecard = scorecard.merge(
    quality_summary[['supplier', 'rejection_pct']],
    on='supplier', how='left'
).fillna({'rejection_pct': 0.0})  # no QIs = no rejections

# Compute dimension scores
scorecard['quality_score']   = (1 - scorecard['rejection_pct'] / 100) * 100
scorecard['ontime_score']    = scorecard['on_time_pct']
scorecard['leadtime_score']  = (np.maximum(0, 1 - scorecard['avg_lead'] / 30) * 100).round(1)

# Composite score
scorecard['score'] = (
    scorecard['quality_score']  * 0.40 +
    scorecard['ontime_score']   * 0.35 +
    scorecard['leadtime_score'] * 0.25
).round(1)

# Grade
def grade(s):
    if s >= 80: return 'A'
    if s >= 60: return 'B'
    if s >= 40: return 'C'
    return 'D'

scorecard['grade'] = scorecard['score'].apply(grade)
scorecard = scorecard.sort_values('score', ascending=False).reset_index(drop=True)

display_cols = ['short', 'grade', 'score', 'quality_score', 'ontime_score', 'leadtime_score', 'avg_lead', 'grn_count']
scorecard[display_cols].rename(columns={
    'short': 'Supplier', 'grade': 'Grade', 'score': 'Score',
    'quality_score': 'Quality (40%)', 'ontime_score': 'On-time (35%)',
    'leadtime_score': 'Lead time (25%)', 'avg_lead': 'Avg lead (d)', 'grn_count': 'GRNs'
})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

grade_colors = {'A': '#10b981', 'B': '#3b82f6', 'C': '#f59e0b', 'D': '#ef4444'}
bar_colors = [grade_colors[g] for g in scorecard['grade']]

bars = ax.barh(scorecard['short'], scorecard['score'], color=bar_colors, height=0.5)
ax.set_xlabel('Composite score (0-100)')
ax.set_title('Supplier Scorecard - FlowForge Industries (Jan-Jun 2026)')
ax.set_xlim(0, 115)

for bar, row in zip(bars, scorecard.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'Grade {row.grade}  |  {row.score}',
            va='center', fontsize=9, fontweight='bold', color=grade_colors[row.grade])

# Grade threshold lines
for threshold, label in [(80, 'A/B'), (60, 'B/C'), (40, 'C/D')]:
    ax.axvline(threshold, color='#9ca3af', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.text(threshold + 0.5, -0.5, label, fontsize=7, color='#9ca3af')

patches = [mpatches.Patch(color=c, label=f'Grade {g}') for g, c in grade_colors.items()]
ax.legend(handles=patches, fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

## 6. Radar Chart - Dimension Breakdown

Visualise each supplier across all three scoring dimensions to see where they win and lose.

In [ ]:
from matplotlib.patches import FancyArrowPatch

categories = ['Quality\n(40%)', 'On-time\n(35%)', 'Lead time\n(25%)']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, axes = plt.subplots(1, len(scorecard), figsize=(14, 3.5), subplot_kw=dict(polar=True))

for ax, (_, row) in zip(axes, scorecard.iterrows()):
    values = [row['quality_score'], row['ontime_score'], row['leadtime_score']]
    values += values[:1]
    color = grade_colors[row['grade']]
    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=7)
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['25', '50', '75', '100'], fontsize=6, color='#9ca3af')
    ax.set_title(f"{row['short']}\nGrade {row['grade']} ({row['score']})",
                 fontsize=9, fontweight='bold', color=color, pad=10)

plt.suptitle('Supplier Performance Radar - All Dimensions', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 7. Reorder Recommendations

The scorecard feeds into reorder logic. For each item type, the primary supplier is identified from the purchase receipt history. The recommendation action depends on the supplier grade:

- **Grade A**: Auto-approve reorder
- **Grade B**: Approve with standard review
- **Grade C**: Approve with quality flag - attach inspection requirements
- **Grade D**: Block auto-reorder - raise resourcing task to find alternative

In [ ]:
# Map each RM item to its primary supplier (most GRNs)
item_supplier = (
    prs_df
    .groupby(['item_code', 'supplier'])
    .size()
    .reset_index(name='grn_count')
    .sort_values('grn_count', ascending=False)
    .groupby('item_code')
    .first()
    .reset_index()[['item_code', 'supplier']]
)

# Merge scorecard grade onto each item
item_supplier = item_supplier.merge(
    scorecard[['supplier', 'short', 'score', 'grade', 'avg_lead']],
    on='supplier', how='left'
)

# Merge item names
item_supplier = item_supplier.merge(
    items_df[['item_code', 'item_name', 'item_group']],
    on='item_code', how='left'
)

# Reorder action based on grade
def reorder_action(grade):
    actions = {
        'A': 'Auto-approve reorder',
        'B': 'Approve with standard review',
        'C': 'Approve with quality flag - attach inspection cert requirement',
        'D': 'BLOCKED - raise resourcing task, find alternative supplier',
    }
    return actions.get(grade, 'Unknown')

item_supplier['reorder_action'] = item_supplier['grade'].apply(reorder_action)

print('Reorder recommendations by item:')
item_supplier[['item_code', 'item_name', 'short', 'grade', 'score', 'avg_lead', 'reorder_action']].to_string(index=False)

In [ ]:
# Summary: items at risk
risk_items = item_supplier[item_supplier['grade'].isin(['C', 'D'])].copy()

print(f'Items requiring reorder review: {len(risk_items)}')
print()
for _, row in risk_items.iterrows():
    print(f"  [{row['grade']}] {row['item_code']:15s} | {row['item_name']:35s} | Primary: {row['short']:12s} (score {row['score']})")
    print(f"       Action: {row['reorder_action']}")
    print()

## 8. Summary and Next Steps

### Scorecard summary

| Supplier | Grade | Score | Key finding |
|---|---|---|---|
| Amrit Metals | A | ~97 | Fastest delivery (5.8d), perfect quality. Preferred vendor for consumables. |
| Purohit Metals | A | ~93 | Consistent 7.8d lead time, 100% on-time, 0 rejections. Top steel vendor. |
| Vishwakarma Tools | B | ~72 | 21d lead time expected for custom tooling. Sole-source - plan ahead. |
| Sudarshan Metal | C | ~48 | EN24 hardness failures in Feb + Apr. 40% rejection rate, 60% on-time. |
| Paschim Steel | D | ~21 | 0% on-time delivery, 40% rejection rate. Dimensional failures Mar + May. |

### Immediate actions

1. **Paschim Steel (D)**: Raise a resourcing task to identify an alternative supplier for SS304 and AL6061 bar stock before the next procurement cycle. Do not auto-create any further POs until a qualified alternative is identified.

2. **Sudarshan Metal (C)**: Require heat treatment certificates with all EN24 billet deliveries. Increase incoming sample size from 5 to 10 for Feb and Apr batches. Re-evaluate after two clean quarters.

3. **Purohit Metals (A)**: Consolidate EN8 and EN19 volume here. If Paschim is replaced for EN24, evaluate whether Purohit can extend to EN24 supply.

### Production pipeline

To run this as a weekly automated scoring job in production:

```python
# 1. Pull latest GRNs and QI records from ERP via API or export
# 2. Run all cells in sequence (or convert to .py script)
# 3. Write scorecard to a dashboard or send as a weekly report
# 4. Compare current scores to previous week - flag any vendors that dropped a grade
# 5. For any reorder trigger, look up the primary vendor grade and route accordingly
```